In [0]:
# Bronze layer implementation

from pyspark.sql import functions as F

base_path = "/Volumes/retail_catalog/raw/landing"

bronze_schema = "retail_catalog.bronze"

In [0]:
# Create a reusable data ingestion function

def ingest_to_bronze(dataset_name):

    source_path = f"{base_path}/{dataset_name}"

    df = spark.read \
        .option("header", True) \
        .option("inferSchema", False) \
        .csv(source_path) \
        .select("*", "_metadata.file_path")

    df = df \
        .withColumnRenamed("file_path", "SourceFile") \
        .withColumn(
            "IngestionTimestamp",
            F.current_timestamp()
        ) \
        .withColumn(
            "SourceSystem",
            F.lit("RetailERP")
        )

    target_table = f"{bronze_schema}.{dataset_name}"

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    print(f"Created table: {target_table}")
    print(f"Records: {df.count()}")

In [0]:
# Execute data ingestion
ingest_to_bronze("customers")
ingest_to_bronze("products")
ingest_to_bronze("sales")

In [0]:
%sql
SHOW TABLES IN retail_catalog.bronze; 
-- Verify Bronze Tables

In [0]:
%sql
SELECT *
FROM retail_catalog.bronze.sales;

In [0]:
# View Schema
bronze_sales = spark.table(
    "retail_catalog.bronze.sales"
)

bronze_sales.printSchema()

In [0]:
%sql
DESCRIBE DETAIL retail_catalog.bronze.sales;

In [0]:
%sql
DESCRIBE HISTORY retail_catalog.bronze.sales;